In [1]:
import pandas as pd

In [3]:
import json
import re
from collections import Counter

INP = "Sifei_taskC_pred.jsonl"  # ← 改成你的文件

# 常见“拒答/不知道”模板（英文）
IDK_PATTERNS = [
    r"\bi\s*(do\s*not|don't)\s*know\b",
    r"\b(i\s*am|i'm)\s*not\s*sure\b",
    r"\bno(t)?\s*enough\s*(info|information)\b",
    r"\b(can(not|'t)|unable\s*to)\s*(answer|provide)\b",
    r"\bthe\s*(document|context|passage)\s*(does\s*not|doesn't)\s*(contain|provide|mention)\b",
    r"\bi\s*(can(not|'t))\s*find\b",
]

idk_re = re.compile("|".join(IDK_PATTERNS), flags=re.IGNORECASE)

def get_pred_text(obj):
    p = obj.get("predictions")
    if isinstance(p, list) and p:
        if isinstance(p[0], dict):
            return str(p[0].get("text", "") or "")
        if isinstance(p[0], str):
            return str(p[0])
    # fallback
    return str(obj.get("prediction", "") or "")

total = 0
idk = 0
examples = []

with open(INP, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        obj = json.loads(line)
        total += 1
        pred = get_pred_text(obj).strip()
        if idk_re.search(pred):
            idk += 1
            if len(examples) < 10:
                examples.append((obj.get("task_id",""), pred[:200]))

print(f"TOTAL: {total}")
print(f"IDK-like: {idk} ({idk/total*100:.2f}%)")
print("\nExamples:")
for tid, t in examples:
    print("-", tid, "=>", t)


TOTAL: 507
IDK-like: 157 (30.97%)

Examples:
- 1dd9e5b32504099bc30a1b5fb64fded5<::>5 => I don't know.
- 132020691f5aa996948ace2b9e4ff27c<::>10 => I don't know.
- 709075ff95308941d87f8c41d96e8a88<::>1 => I don't know.
- 514deeb00eb0aef56f5113bce69240ab<::>7 => I don't know.

The provided documents describe the historical legal barriers (DOMA) that previously prevented same-sex couples from accessing family-based immigration benefits like spousal petitions a
- 1b8731b16f93813ba2d9cd06b42886df<::>7 => I don't know.
- 5c6fe37fcb387a5d23c98581b3bb44db<::>9 => I don't know.
- 6da5ffc018f9a4a526debf9566f6a64d<::>6 => I don't know.
- 81595fbce3b3bc2ab66ebebc01346539<::>7 => I don't know.
- 13d4dd7172433639010710de00dbe10e<::>6 => I don't know.
- b97a2b7f509c5e21ca6ee401b76896b4<::>4 => I don't know. The provided documents do not list the specific hardware and software requirements for SAP HANA. They direct the user to consult external official sources such as the SAP Product Availab


In [7]:
df = pd.read_json("Sifei_taskC_pred.jsonl", lines=True)
df

,conversation_id,task_id,Collection,input,contexts,predictions
0,18ef26058d321c5d96ca3ebf8117789e,18ef26058d321c5d96ca3ebf8117789e<::>7,mt-rag-fiqa-beir-elser-512-100-20240501,"[{'speaker': 'user', 'text': 'How to pay with ...","[{'document_id': '481165-0-886', 'text': '""The...",[{'text': 'Current EV battery degradation and ...
1,b3b321e9ea81d1d90e528f85fff72d63,b3b321e9ea81d1d90e528f85fff72d63<::>8,mt-rag-ibmcloud-elser-512-100-20240502,"[{'speaker': 'user', 'text': 'Can you summariz...","[{'document_id': 'ibmcld_05430-7-1528', 'text'...","[{'text': 'No, IBM Cloud toolchain is availabl..."
2,72013190593248ab20cca034f21ce38a,72013190593248ab20cca034f21ce38a<::>4,mt-rag-ibmcloud-elser-512-100-20240502,"[{'speaker': 'user', 'text': 'What are the dif...","[{'document_id': 'ibmcld_02998-3401-4882', 'te...",[{'text': 'The steps involved include creating...
3,2f671f98cc9ba4051f126197b0039622,2f671f98cc9ba4051f126197b0039622<::>1,mt-rag-clapnq-elser-512-100-20240503,"[{'speaker': 'user', 'text': 'what is the diff...","[{'document_id': '806502664_965-1711-0-746', '...",[{'text': 'Primary sources are original materi...
4,fa60731970330a3f86312cd7c38762c0,fa60731970330a3f86312cd7c38762c0<::>2,mt-rag-fiqa-beir-elser-512-100-20240501,"[{'speaker': 'user', 'text': ' What's the diff...","[{'document_id': '471123-0-798', 'text': '""Mar...",[{'text': 'Market cap is the total market valu...
...,...,...,...,...,...,...
502,5a62ecb3558b31040abc6515bf1d8ca1,5a62ecb3558b31040abc6515bf1d8ca1<::>2,mt-rag-ibmcloud-elser-512-100-20240502,"[{'speaker': 'user', 'text': 'Can you tell me ...","[{'document_id': 'ibmcld_07578-757483-759510',...",[{'text': 'I don't know.'}]
503,bfdf6b5397ba1a9b014e7a4c1e35bc89,bfdf6b5397ba1a9b014e7a4c1e35bc89<::>2,mt-rag-clapnq-elser-512-100-20240503,"[{'speaker': 'user', 'text': 'when was star sp...","[{'document_id': '815214581_627-1215-0-588', '...",[{'text': 'I don't know.'}]
504,0d4c0bd5c782a8e57c797df60c1a7004,0d4c0bd5c782a8e57c797df60c1a7004<::>2,mt-rag-ibmcloud-elser-512-100-20240502,"[{'speaker': 'user', 'text': 'How do I copy th...","[{'document_id': 'ibmcld_00519-5675-7405', 'te...",[{'text': 'I don't know.'}]
505,767c3b388e9f9ca8b26e33a744506d54,767c3b388e9f9ca8b26e33a744506d54<::>3,mt-rag-clapnq-elser-512-100-20240503,"[{'speaker': 'user', 'text': 'aws', 'metadata'...",[{'document_id': '846470453_11870-12030-0-160'...,[{'text': 'I don't know.'}]
